# Explore and Discover

This notebook is your playground. You have already seen what one galaxy looks like in Notebook 4 — now we will use the same SKIRT data to explore real astrophysics questions.

Each experiment asks you to change something and observe what happens. There are guiding questions after each one, but do not just answer and move on — try to understand *why* you see what you see.

Available galaxies (all pre-downloaded):

| ID | Character | Stellar mass | Star-formation rate |
|---|---|---|---|
| `553837` | Star-forming spiral (your galaxy from Notebook 4) | ~2.5 × 10¹⁰ M☉ | 1.7 M☉/yr |
| `503987` | Passive / quenched galaxy | ~4.2 × 10¹⁰ M☉ | 0.07 M☉/yr |
| `184931` | Massive starburst | ~5.4 × 10¹¹ M☉ | 38 M☉/yr |
| `618580` | Small star-forming galaxy | ~5 × 10⁹ M☉ | 1.0 M☉/yr |

In [ ]:
# Run this cell first — it installs everything you need.

%pip install -q astropy requests
print("All packages ready!")

In [ ]:
import sys, os
sys.path.insert(0, ".")
from helpers import (
    download_skirt_galaxy, extract_skirt_tar,
    skirt_file, load_fits_image,
    plot_skirt_image, plot_skirt_multiwave,
    plot_skirt_rgb, plot_skirt_property_maps,
    SKIRT_FILTERS,
)
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

# ============================================================
# PASTE YOUR API KEY HERE
# ============================================================
API_KEY = "693fc85106de8da5bf5a9eebf34c3fc5"
# ============================================================

print("Ready!")

In [ ]:
# Make sure all four galaxies are downloaded and extracted.
# This cell checks for existing data and only downloads what is missing.

for gid in ["553837", "503987", "184931", "618580"]:
    tar_path, outdir = download_skirt_galaxy(gid, api_key=API_KEY)
    extract_skirt_tar(tar_path, outdir)

print("\nAll galaxies ready!")

---

## Experiment 1 — Does Viewing Angle Matter?

A real astronomer can only see a galaxy from one direction. We happen to be looking at the Milky Way edge-on (because we are inside it), but most external galaxies are at random orientations.

The SKIRT atlas provides **five viewing orientations** (O1–O5) for each galaxy:
- **O1** — approximately face-on (disk is almost circular)
- **O3** — approximately edge-on (disk appears as a thin stripe)
- **O2, O4, O5** — intermediate angles

In this experiment you will look at galaxy `553837` from all five angles in the same filter, and think about what an astronomer would conclude from each view.

In [ ]:
# Galaxy 553837 seen from all five orientations — LSST r-band

GALAXY_ID = "553837"
FILTER    = "LSST_r"

# ── Zoom controls ───────────────────────────────────────
FOV_KPC = 160.0
XLIM    = None    # e.g. (-50, 50)
YLIM    = None
# ────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 5, figsize=(20, 5), facecolor="black")
half = FOV_KPC / 2

for ax, orient in zip(axes, ["O1", "O2", "O3", "O4", "O5"]):
    img = load_fits_image(skirt_file(GALAXY_ID, orient, FILTER))
    arr = np.log10(np.clip(img, 1e-6, None))
    ax.imshow(arr, origin="lower", cmap="magma",
              extent=[-half, half, -half, half])
    ax.set_title(f"Orientation {orient}", color="white", fontsize=11)
    ax.set_xlabel("x (kpc)", color="white", fontsize=8)
    ax.set_ylabel("y (kpc)", color="white", fontsize=8)
    ax.tick_params(colors="white", labelsize=7)
    ax.set_facecolor("black")
    for spine in ax.spines.values():
        spine.set_edgecolor("white")
    if XLIM: ax.set_xlim(XLIM)
    if YLIM: ax.set_ylim(YLIM)

fig.patch.set_facecolor("black")
fig.suptitle(f"Galaxy {GALAXY_ID} — {FILTER} — five viewing angles",
             color="white", fontsize=13)
plt.tight_layout()
plt.show()

### Questions for Experiment 1

1. Looking at the five orientations, which one would you classify as *face-on* and which as *edge-on*?

2. In the edge-on view, can you see a dark lane running through the middle of the galaxy?
   What is causing it?
   *(Hint: think about where the dust lives in a spiral galaxy — mostly in the disk — and what happens to the light from stars behind that dust.)*

3. If you were an astronomer and could only observe one image of this galaxy, would your estimate of its size depend on which orientation you happened to observe it from?

4. Try changing `FILTER` in the cell above to `"GALEX_FUV"` or `"2MASS_Ks"`. Does the edge-on dust lane look the same in all filters?

---

## Experiment 2 — How Much Does Dust Hide?

We saw in Notebook 4 that dust absorbs light — especially UV. But *how much* does it actually hide, and does the effect depend on where you look inside the galaxy?

In this experiment you will:
1. Compare images with and without dust across three wavelength regions
2. Compute the **attenuation map** — a pixel-by-pixel measure of how much dust is dimming each part of the galaxy
3. Check whether the dust effect is uniform across the galaxy or concentrated in certain regions

In [ ]:
# Compute attenuation maps in three filters
# Attenuation (magnitudes) = -2.5 * log10(flux_with_dust / flux_no_dust)
# Positive values mean the dust is dimming the light.

GALAXY_ID = "553837"
ORIENT    = "O1"

attenuation_filters = [
    ("GALEX_FUV", "GALEX FUV\n(~1500 Å — far UV)",  "violet"),
    ("Johnson_V",  "Johnson V\n(~5500 Å — optical)",  "limegreen"),
    ("2MASS_Ks",   "2MASS Ks\n(~2.2 μm — near-IR)",  "tomato"),
]

# ── Zoom controls ───────────────────────────────────────
FOV_KPC = 160.0
XLIM    = None
YLIM    = None
# ────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor="black")
half = FOV_KPC / 2

for ax, (fname, label, colour) in zip(axes, attenuation_filters):
    with_dust = load_fits_image(skirt_file(GALAXY_ID, ORIENT, fname))
    no_dust   = load_fits_image(skirt_file(GALAXY_ID, ORIENT, fname, nodust=True))

    # Attenuation in magnitudes (positive = dust is dimming)
    att = -2.5 * np.log10(
        np.clip(with_dust, 1e-12, None) / np.clip(no_dust, 1e-12, None)
    )
    att = np.clip(att, 0, 10)   # remove unphysical negative values

    im = ax.imshow(att, origin="lower", cmap="hot",
                   extent=[-half, half, -half, half],
                   vmin=0)
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.05)
    cbar = fig.colorbar(im, cax=cax)
    cbar.set_label("Attenuation (mag)", color="white", fontsize=9)
    cbar.ax.yaxis.set_tick_params(color="white")
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color="white")

    mean_att = att[att > 0.01].mean() if (att > 0.01).any() else 0
    ax.set_title(f"{label}\nmean A = {mean_att:.2f} mag",
                 color=colour, fontsize=10, pad=4)
    ax.set_xlabel("x (kpc)", color="white", fontsize=8)
    ax.set_ylabel("y (kpc)", color="white", fontsize=8)
    ax.tick_params(colors="white", labelsize=7)
    ax.set_facecolor("black")
    for spine in ax.spines.values():
        spine.set_edgecolor("white")
    if XLIM: ax.set_xlim(XLIM)
    if YLIM: ax.set_ylim(YLIM)

fig.patch.set_facecolor("black")
fig.suptitle(f"Galaxy {GALAXY_ID} — Dust Attenuation Maps",
             color="white", fontsize=13)
plt.tight_layout()
plt.show()

print("Attenuation = how many magnitudes of light the dust is absorbing.")
print("1 magnitude ≈ a factor of 2.5 in brightness lost to dust.")

### Questions for Experiment 2

1. Compare the three attenuation maps. In which filter is the attenuation greatest? Does this match what you expected?

2. Look at *where* the attenuation is highest inside the galaxy. Is it uniform, or concentrated somewhere?
   *(Hint: compare with the dust mass map from Notebook 4 — do they match?)*

3. If an astronomer measures this galaxy's UV brightness from a real telescope and uses it to estimate the star-formation rate, but forgets to correct for dust — will they **over-estimate** or **under-estimate** the true rate?

4. Do you think an older, passive galaxy (one with very little gas and dust) would have the same attenuation? Why or why not?

---

## Experiment 3 — Star-forming vs Passive: Two Different Galaxies

Not all galaxies are actively forming new stars. Some galaxies — called *passive*, *quenched*, or *red-and-dead* — stopped forming stars billions of years ago. Their stellar populations are dominated by old, cool, red stars.

In this experiment we compare two galaxies:
- **Galaxy 553837** — still forming stars (SFR = 1.7 M☉/yr)
- **Galaxy 503987** — quenched (SFR = 0.07 M☉/yr, essentially zero)

In [ ]:
# Compare the two galaxies across UV, optical, and near-IR in a grid

GAL_A   = "553837"   # star-forming spiral
GAL_B   = "503987"   # passive / quenched
ORIENT  = "O1"

compare_bands = [
    ("GALEX_FUV", "FUV (~1500 Å)"),
    ("Johnson_B",  "B-band (~4400 Å)"),
    ("LSST_r",     "r-band (~6200 Å)"),
    ("2MASS_Ks",   "Ks-band (~2.2 μm)"),
]

# ── Zoom controls ───────────────────────────────────────
FOV_KPC = 160.0
XLIM    = None
YLIM    = None
# ────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 4, figsize=(18, 9), facecolor="black")
half = FOV_KPC / 2

labels_ab = [(GAL_A, "steelblue"), (GAL_B, "tomato")]

for row, (gid, gcolour) in enumerate(labels_ab):
    for col, (fname, flabel) in enumerate(compare_bands):
        ax = axes[row, col]
        img = load_fits_image(skirt_file(gid, ORIENT, fname))
        arr = np.log10(np.clip(img, 1e-6, None))
        ax.imshow(arr, origin="lower", cmap="magma",
                  extent=[-half, half, -half, half])
        title = f"{flabel}" if row == 0 else ""
        ylabel = f"Galaxy {gid}" if col == 0 else ""
        ax.set_title(title, color="white", fontsize=10)
        ax.set_ylabel(ylabel, color=gcolour, fontsize=10)
        ax.set_xlabel("x (kpc)", color="white", fontsize=8)
        ax.tick_params(colors="white", labelsize=7)
        ax.set_facecolor("black")
        for spine in ax.spines.values():
            spine.set_edgecolor(gcolour if col == 0 else "white")
        if XLIM: ax.set_xlim(XLIM)
        if YLIM: ax.set_ylim(YLIM)

fig.patch.set_facecolor("black")
fig.suptitle(
    f"Two galaxies compared — top: {GAL_A} (star-forming)   bottom: {GAL_B} (quenched)",
    color="white", fontsize=12,
)
plt.tight_layout()
plt.show()

In [ ]:
# Compare the physical property maps of both galaxies

print(f"=== Galaxy {GAL_A} (star-forming) ===")
plot_skirt_property_maps(GAL_A, ORIENT, fov_kpc=FOV_KPC, xlim=XLIM, ylim=YLIM)

print(f"\n=== Galaxy {GAL_B} (quenched) ===")
plot_skirt_property_maps(GAL_B, ORIENT, fov_kpc=FOV_KPC, xlim=XLIM, ylim=YLIM)

In [ ]:
# Make RGB colour images of both galaxies for a direct visual comparison

for gid, label, stretch in [(GAL_A, "star-forming", 0.05),
                              (GAL_B, "quenched",     0.03)]:
    r = load_fits_image(skirt_file(gid, ORIENT, "LSST_r"))
    g = load_fits_image(skirt_file(gid, ORIENT, "LSST_g"))
    b = load_fits_image(skirt_file(gid, ORIENT, "Johnson_U"))
    plot_skirt_rgb(
        r, g, b,
        title   = f"Galaxy {gid} ({label}) — R=LSST_r  G=LSST_g  B=Johnson_U",
        stretch = stretch,
        fov_kpc = FOV_KPC,
        xlim    = XLIM,
        ylim    = YLIM,
    )

### Questions for Experiment 3

1. Look at the UV (FUV) images. Which galaxy is brighter in UV? Why?
   *(Hint: UV comes from hot, young, massive stars — which only exist if stars are forming now.)*

2. Look at the near-infrared (Ks) images. Are they more similar to each other than the UV images? Why?
   *(Hint: near-IR comes from cool, old, low-mass stars — which exist in both galaxies.)*

3. Look at the colour RGB images. Which galaxy looks bluer and which looks redder overall? What does the colour tell you about each galaxy's history?

4. Look at the physical property maps. How do the **dust mass** maps compare between the two galaxies? Why would a quenched galaxy have less dust?
   *(Hint: dust is made from heavy elements ejected by dying stars, and is destroyed by radiation and stellar winds — in a galaxy with no young massive stars, the balance shifts.)*

5. If you were an astronomer looking at a real sky image and saw these two galaxies side by side, how would you classify them before doing any detailed analysis?

---

## Experiment 4 — A Mini Galaxy Survey

Real survey telescopes (like the Vera Rubin Observatory with its LSST camera) photograph billions of galaxies simultaneously. In each image, galaxies come in all shapes and sizes and at random orientations.

In this experiment you build a small version of that survey: you will look at all four available TNG50-1 galaxies in a single filter and compare them. Think of this as observing four galaxies in the same "patch of sky".

In [ ]:
# Show all four galaxies in LSST r-band — same scale so sizes are comparable

survey_galaxies = [
    ("553837", "Star-forming spiral\nM★ = 2.5×10¹⁰ M☉, SFR = 1.7 M☉/yr", "steelblue"),
    ("503987", "Passive / quenched\nM★ = 4.2×10¹⁰ M☉, SFR = 0.07 M☉/yr", "tomato"),
    ("184931", "Massive starburst\nM★ = 5.4×10¹¹ M☉, SFR = 38 M☉/yr",    "gold"),
    ("618580", "Small star-forming\nM★ = 5×10⁹ M☉, SFR = 1.0 M☉/yr",     "mediumseagreen"),
]

SURVEY_FILTER = "LSST_r"   # ← try changing this
ORIENT        = "O1"

# ── Zoom controls ───────────────────────────────────────
FOV_KPC = 160.0
XLIM    = None
YLIM    = None
# ────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 4, figsize=(20, 6), facecolor="black")
half = FOV_KPC / 2

for ax, (gid, label, colour) in zip(axes, survey_galaxies):
    img = load_fits_image(skirt_file(gid, ORIENT, SURVEY_FILTER))
    arr = np.log10(np.clip(img, 1e-6, None))
    ax.imshow(arr, origin="lower", cmap="magma",
              extent=[-half, half, -half, half])
    ax.set_title(f"ID {gid}\n{label}", color=colour, fontsize=9, pad=5)
    ax.set_xlabel("x (kpc)", color="white", fontsize=8)
    ax.set_ylabel("y (kpc)", color="white", fontsize=8)
    ax.tick_params(colors="white", labelsize=7)
    ax.set_facecolor("black")
    for spine in ax.spines.values():
        spine.set_edgecolor("white")
    if XLIM: ax.set_xlim(XLIM)
    if YLIM: ax.set_ylim(YLIM)

fig.patch.set_facecolor("black")
fig.suptitle(f"Four TNG50-1 galaxies — {SURVEY_FILTER}",
             color="white", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Now compare the same four galaxies in UV vs near-IR side by side
# This is the astronomer's classic "colour comparison"

fig, axes = plt.subplots(2, 4, figsize=(20, 10), facecolor="black")
half = FOV_KPC / 2

for col, (gid, label, colour) in enumerate(survey_galaxies):
    for row, (fname, band_label, cmap) in enumerate([
        ("GALEX_FUV", "GALEX FUV  (hot young stars)", "magma"),
        ("2MASS_Ks",  "2MASS Ks   (all old stars)",   "inferno"),
    ]):
        ax = axes[row, col]
        img = load_fits_image(skirt_file(gid, ORIENT, fname))
        arr = np.log10(np.clip(img, 1e-6, None))
        ax.imshow(arr, origin="lower", cmap=cmap,
                  extent=[-half, half, -half, half])
        if row == 0:
            ax.set_title(f"ID {gid}\n{label.split(chr(10))[0]}",
                         color=colour, fontsize=9, pad=4)
        ax.set_ylabel(band_label, color="white", fontsize=8)
        ax.set_xlabel("x (kpc)", color="white", fontsize=7)
        ax.tick_params(colors="white", labelsize=6)
        ax.set_facecolor("black")
        for spine in ax.spines.values():
            spine.set_edgecolor("white")
        if XLIM: ax.set_xlim(XLIM)
        if YLIM: ax.set_ylim(YLIM)

fig.patch.set_facecolor("black")
fig.suptitle("Four galaxies — UV (top) vs near-infrared (bottom)",
             color="white", fontsize=13)
plt.tight_layout()
plt.show()

print("\nKey question: which galaxy changes the MOST between UV and near-IR?")
print("What does that tell you about its star formation?")

### Questions for Experiment 4

1. In the r-band survey image, which galaxy looks biggest on the sky? Does bigger always mean more massive?
   *(The massive starburst 184931 has 20× the stellar mass of the small galaxy 618580 — can you tell just from the image?)*

2. In the UV vs near-IR comparison, which galaxy is brightest in UV relative to its near-IR brightness?
   What does a high UV-to-infrared ratio tell you about a galaxy?

3. The passive galaxy (503987) has almost no UV emission. If you found a galaxy like this in real telescope data, what would you conclude about its history?

4. Try changing `SURVEY_FILTER` to `"GALEX_FUV"`, `"Johnson_B"`, or `"WISE_W1"`. Which filter gives you the most information about which galaxies are actively forming stars?

---

## What You Have Accomplished

You have now explored genuine astrophysics using professional-grade data:

- Seen how a single galaxy looks different depending on viewing angle, wavelength, and dust
- Measured dust attenuation quantitatively — in magnitudes, like a real astronomer
- Compared a star-forming galaxy with a passive one and understood *why* they differ
- Built a small galaxy survey and found that UV and infrared reveal completely different physics

This is the kind of multi-wavelength analysis that Sophie does in her research — comparing what simulations predict with what real telescopes observe.

---

*If you want to explore further, try changing the galaxy IDs in any experiment above, or try different filter combinations for the RGB images. There are no wrong answers — every combination reveals something new.*